## Notebook 02 — Full Data Audit on Merged Train Dataset

---

This notebook performs a complete audit on the merged training data:
1. Total memory usage in MB
2. Total number of columns
3. Column type breakdown (numeric / object / boolean)
4. Missing value percentage per column
5. Count of columns with >80% and >50% missing
6. Top 20 columns by highest missing percentage
7. Columns with only 1 unique value (zero variance)

> **Prerequisite:** Run `01_data_loading_and_merging.ipynb` first, or use the data loading cell below.

---
## 📦 Step 0 — Import Libraries & Load Merged Train Data

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../Datasets/'

print('Loading and merging datasets...')

train_transaction = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(DATA_PATH + 'train_identity.csv')

train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')

# Free up raw dataframes to save memory
del train_transaction, train_identity

print(f'✅ Merged train loaded — shape: {train.shape}')

Loading and merging datasets...


✅ Merged train loaded — shape: (590540, 434)


---
## 💾 Step 1 — Total Memory Usage (MB)

In [2]:
mem_mb = train.memory_usage(deep=True).sum() / (1024 ** 2)

print('=' * 50)
print('           MEMORY USAGE SUMMARY                ')
print('=' * 50)
print(f'  Total Memory Usage  : {mem_mb:.2f} MB')
print(f'  Rows                : {train.shape[0]:,}')
print(f'  Columns             : {train.shape[1]:,}')
print(f'  Avg Memory per Col  : {mem_mb / train.shape[1]:.2f} MB')
print('=' * 50)

           MEMORY USAGE SUMMARY                
  Total Memory Usage  : 2567.09 MB
  Rows                : 590,540
  Columns             : 434
  Avg Memory per Col  : 5.91 MB


---
## 🔢 Step 2 & 3 — Total Columns & Type Breakdown

In [3]:
total_cols   = train.shape[1]
num_cols     = train.select_dtypes(include=['number']).shape[1]
obj_cols     = train.select_dtypes(include=['object']).shape[1]
bool_cols    = train.select_dtypes(include=['bool']).shape[1]
other_cols   = total_cols - num_cols - obj_cols - bool_cols

print('=' * 50)
print('         COLUMN TYPE BREAKDOWN                 ')
print('=' * 50)
print(f'  Total Columns    : {total_cols}')
print('-' * 50)
print(f'  Numeric          : {num_cols}  ({num_cols/total_cols*100:.1f}%)')
print(f'  Object (string)  : {obj_cols}  ({obj_cols/total_cols*100:.1f}%)')
print(f'  Boolean          : {bool_cols}  ({bool_cols/total_cols*100:.1f}%)')
if other_cols > 0:
    print(f'  Other            : {other_cols}  ({other_cols/total_cols*100:.1f}%)')
print('=' * 50)

# Show actual dtypes summary
print('\nData Types Detail:')
print(train.dtypes.value_counts().to_string())

         COLUMN TYPE BREAKDOWN                 
  Total Columns    : 434
--------------------------------------------------
  Numeric          : 403  (92.9%)
  Object (string)  : 31  (7.1%)
  Boolean          : 0  (0.0%)

Data Types Detail:
float64    399
object      31
int64        4


---
## ❓ Step 4 — Missing Value Percentage Per Column

In [4]:
# Calculate missing stats for every column
missing_count = train.isnull().sum()
missing_pct   = (missing_count / len(train) * 100).round(2)

missing_df = pd.DataFrame({
    'Column'          : missing_count.index,
    'Missing Count'   : missing_count.values,
    'Missing %'       : missing_pct.values,
    'Dtype'           : train.dtypes.values
}).sort_values('Missing %', ascending=False).reset_index(drop=True)

# Only show columns that actually have missing values
missing_nonzero = missing_df[missing_df['Missing %'] > 0]

print('=' * 55)
print('         MISSING VALUE OVERVIEW                   ')
print('=' * 55)
print(f'  Total Columns            : {total_cols}')
print(f'  Columns with NO missing  : {(missing_pct == 0).sum()}')
print(f'  Columns with ANY missing : {(missing_pct > 0).sum()}')
print('=' * 55)

         MISSING VALUE OVERVIEW                   
  Total Columns            : 434
  Columns with NO missing  : 52
  Columns with ANY missing : 382


---
## 📊 Step 5 — Columns with >80% and >50% Missing

In [5]:
cols_80 = (missing_pct > 80).sum()
cols_50 = (missing_pct > 50).sum()
cols_20 = (missing_pct > 20).sum()

print('=' * 55)
print('      MISSING DATA THRESHOLD SUMMARY              ')
print('=' * 55)
print(f'  Columns with > 80% missing  : {cols_80:>5}  ({cols_80/total_cols*100:.1f}% of all cols)')
print(f'  Columns with > 50% missing  : {cols_50:>5}  ({cols_50/total_cols*100:.1f}% of all cols)')
print(f'  Columns with > 20% missing  : {cols_20:>5}  ({cols_20/total_cols*100:.1f}% of all cols)')
print('=' * 55)
print(f'\n  ⚠️  {cols_80} columns are candidates for DROP (>80% missing)')
print(f'  ⚠️  {cols_50} columns need imputation strategy (>50% missing)')

      MISSING DATA THRESHOLD SUMMARY              
  Columns with > 80% missing  :    74  (17.1% of all cols)
  Columns with > 50% missing  :   214  (49.3% of all cols)
  Columns with > 20% missing  :   252  (58.1% of all cols)

  ⚠️  74 columns are candidates for DROP (>80% missing)
  ⚠️  214 columns need imputation strategy (>50% missing)


---
## 🏆 Step 6 — Top 20 Columns by Highest Missing Percentage

In [6]:
top20_missing = missing_df.head(20)

print('=' * 65)
print('     TOP 20 COLUMNS WITH HIGHEST MISSING PERCENTAGE         ')
print('=' * 65)
print(f'{"Rank":<5} {"Column":<25} {"Missing Count":>15} {"Missing %":>10} {"Dtype":>10}')
print('-' * 65)
for i, row in top20_missing.iterrows():
    print(f'{i+1:<5} {row["Column"]:<25} {int(row["Missing Count"]):>15,} {row["Missing %"]:>9.2f}% {str(row["Dtype"]):>10}')
print('=' * 65)

     TOP 20 COLUMNS WITH HIGHEST MISSING PERCENTAGE         
Rank  Column                      Missing Count  Missing %      Dtype
-----------------------------------------------------------------
1     id_24                             585,793     99.20%    float64
2     id_25                             585,408     99.13%    float64
3     id_26                             585,377     99.13%    float64
4     id_21                             585,381     99.13%    float64
5     id_08                             585,385     99.13%    float64
6     id_07                             585,385     99.13%    float64
7     id_27                             585,371     99.12%     object
8     id_23                             585,371     99.12%     object
9     id_22                             585,371     99.12%    float64
10    dist2                             552,913     93.63%    float64
11    D7                                551,623     93.41%    float64
12    id_18                      

---
## 🔍 Step 7 — Columns with Only 1 Unique Value (Zero Variance)

In [7]:
unique_counts = train.nunique(dropna=False)
single_unique = unique_counts[unique_counts == 1]

print('=' * 55)
print('      ZERO-VARIANCE COLUMNS (only 1 unique value)  ')
print('=' * 55)

if len(single_unique) == 0:
    print('  ✅ No columns with only 1 unique value found.')
else:
    print(f'  Found {len(single_unique)} zero-variance column(s):\n')
    print(f'  {"Column":<30} {"Unique Value"}')
    print('  ' + '-' * 45)
    for col in single_unique.index:
        val = train[col].dropna().unique()
        val_str = str(val[0]) if len(val) > 0 else 'NaN only'
        print(f'  {col:<30} {val_str}')

print('=' * 55)

      ZERO-VARIANCE COLUMNS (only 1 unique value)  
  ✅ No columns with only 1 unique value found.


---
## 🗂️ Bonus — Full Missing Value Table (All Columns with Missing Data)

In [8]:
pd.set_option('display.max_rows', 200)

display_df = missing_nonzero[['Column', 'Missing Count', 'Missing %', 'Dtype']].copy()
display_df.index = range(1, len(display_df) + 1)
print(f'Showing all {len(display_df)} columns that have at least 1 missing value:\n')
display_df

Showing all 382 columns that have at least 1 missing value:



,Column,Missing Count,Missing %,Dtype
1,id_24,585793,99.20,float64
2,id_25,585408,99.13,float64
3,id_26,585377,99.13,float64
4,id_21,585381,99.13,float64
5,id_08,585385,99.13,float64
...,...,...,...,...
378,V116,314,0.05,float64
379,V115,314,0.05,float64
380,V114,314,0.05,float64
381,V113,314,0.05,float64


---
## ✅ Audit Summary

| Step | Description | Status |
|------|------------|--------|
| 1 | Total memory usage (MB) | ✅ |
| 2 | Total number of columns | ✅ |
| 3 | Column type breakdown | ✅ |
| 4 | Missing % per column | ✅ |
| 5 | Columns with >80% and >50% missing | ✅ |
| 6 | Top 20 highest missing columns | ✅ |
| 7 | Zero-variance columns | ✅ |

---
> **Next Step:** `03_exploratory_data_analysis.ipynb` — Distributions, correlations, and visualizations.